# NHS autism pathway DES — runner guide

Tutorial for **`des.runners`**: `single_run` → `multiple_replication` → **Run 1 / 2 / 3**.

| Layer | Function | Purpose |
|-------|----------|---------|
| Atomic | `single_run` | One replication → patients, capacity, model_params, `RunReport` |
| Stochastic | `multiple_replication` | Many reps + `summarise_replications` |
| Run 1 | `run1` | Calibrate horizon **T\*** to provider targets |
| Run 2 | `run2` | Baseline CIs at **T\*** |
| Run 3 | `run3` | Policy switch at **T\***, backlog decay vs control |

Set **`DEMO_FAST = True`** (default) for a quick run; **`False`** for ~18-year PTL calibration.

**Streamlit UI:** `./run_streamlit.sh` or deploy via **`DEPLOY_STREAMLIT.md`**.

KPI glossary: **`GLOSSARY.md`**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from des.audit import Audit
from des.experiment import Experiment
from des.run_report import KPI_LABELS, kpi_snapshot
from des.runners import multiple_replication, run1, run2, run3, single_run, summarise_replications


In [ ]:
DEMO_FAST = True
N_REP = 3 if DEMO_FAST else 5
N_JOBS = 1
FLOW_WINDOW_DAYS = 365.0
DEFAULT_RND_SET = 42
N_STREAMS = 15
WARM_UP_PERIOD = 0

MONTHLY_REFERRALS = 41
WORKING_DAYS_PER_MONTH = (52 * 5) / 12
IAT_WEEKDAY = 1 / (MONTHLY_REFERRALS / WORKING_DAYS_PER_MONTH)

NOTEBOOK_EXPERIMENT_PARAMETERS = {
    "iat": IAT_WEEKDAY,
    "pct_referral_rejected": 0.369,
    "pct_admin_removal": 0.10,
    "assessment_appointment_counts": [2, 3, 4, 5],
    "assessment_appointment_probs": [0.40, 0.30, 0.20, 0.10],
    "assessment_gap_days": 7,
    "duration_assessment": [2.0, 2.5, 3.0],
    "pct_diagnosis": 0.75,
    "pct_virtual_support": 0.30,
    "workshop_group_size": 8,
    "workshop_num_sessions": 6,
    "workshop_session_interval_weeks": 1,
    "workshop_max_wait_days": 28,
    "duration_workshop_session": [2, 3, 4],
    "workforce_hours_workshop_session": 2.0,
    "workforce_hours_per_day": 7,
}


def make_experiment(name: str) -> Experiment:
    return Experiment(
        audit=Audit(),
        random_number_set=DEFAULT_RND_SET,
        n_streams=N_STREAMS,
        use_fixed_seed=True,
        scenario_name=name,
        **NOTEBOOK_EXPERIMENT_PARAMETERS,
    )


## 1. `single_run`

Returns **`patients`**, **`capacity`**, **`model_params`**, **`report`**.

| Output | Captured how |
|--------|----------------|
| `patients` | `Patient` processes → `audit.update_patient` → `finalize()` |
| `capacity` | Weekday workforce → `audit.record_capacity_day` |
| `model_params` | `Experiment.to_kwargs()` after run |
| `report` | `build_run_report(patients, capacity, sim_end=...)` (derived KPIs) |


In [ ]:
experiment = make_experiment("demo_single_run")

patients, capacity, model_params, report = single_run(
    experiment,
    rep=0,
    run_length=730 if DEMO_FAST else 365 * 3,
    warm_up=WARM_UP_PERIOD,
    flow_window_days=FLOW_WINDOW_DAYS,
)

headline = kpi_snapshot(report)
print(f"Horizon {report.sim_end:.0f} d | patients {len(patients):,} | backlog {headline.get('backlog_patients_at_horizon')}")
report.pathway_funnel.head(8)


In [ ]:
patients.head()
capacity.head()
pd.Series(model_params)


## 2. `multiple_replication`


In [ ]:
multi = multiple_replication(
    make_experiment("demo_multi"),
    n_reps=N_REP,
    run_length=730 if DEMO_FAST else 365 * 5,
    warm_up=0,
    flow_window_days=FLOW_WINDOW_DAYS,
    n_jobs=N_JOBS,
)
summarise_replications(multi).rtt_waits_stock.query('stat == "mean_wait_days"')


## 3. Run 1 — **T\***


In [ ]:
TARGET_PTL = 800.0 if DEMO_FAST else 2835.0
run1_result = run1(
    make_experiment("run1"),
    targets={"waiting_list_size_all_in_system": TARGET_PTL},
    max_period_days=5 * 365 if DEMO_FAST else 18 * 365,
    step_days=365,
    min_period_days=365,
    match_tolerance=0.15 if DEMO_FAST else 0.05,
    flow_window_days=FLOW_WINDOW_DAYS,
)
T_STAR = float(run1_result["optimal_matching_period_days"])
print(f"T*={T_STAR:.0f} d matched={run1_result['matched']}")
run1_result["history"][["horizon_days", "backlog_patients_at_horizon", "mape", "matched"]].round(3)


## 4. Run 2 — baseline at **T\***


In [ ]:
run2_result = run2(
    make_experiment("run2"),
    matching_period_days=T_STAR,
    n_reps=N_REP,
    flow_window_days=FLOW_WINDOW_DAYS,
    n_jobs=N_JOBS,
)
run2_result["kpi_snapshots"][["rep", "backlog_patients_at_horizon", "backlog_mean_wait_days"]]


## 5. Run 3 — policy decay


In [ ]:
run3_result = run3(
    make_experiment("run3"),
    matching_period_days=T_STAR,
    decay_period_days=365.0,
    policy_overrides={"workforce_hours_per_day": 9.0},
    n_reps=max(2, N_REP),
    include_control=True,
)
run3_result.get("comparison")
